# CS336 Spring 2026 - 15 Assignment 1 知识整合与 Resource Accounting
----
> 这是 Jupyter 学习阶段的总结章节，不是作业代码或练习题。它把 01–14 对应到 Assignment 1 的完整训练栈，并补足 **参数量、FLOPs、显存预算** 的基础知识。

官方入口：
- 课程主页：https://cs336.stanford.edu/
- Assignment 1 仓库：https://github.com/stanford-cs336/assignment1-basics

本章的所有代码用于数学核对、接口说明或轻量演示；具体的测试 adapter 和签名以正式克隆时的官方仓库为准。


## 1. Assignment 1 的整体数据流

```text
01–04 Unicode, bytes, pretokenization, special tokens
    ↓
05–07 train BPE → vocab, merges
    ↓
08 encode/decode → token ids (B,T)
    ↓
09 Embedding + Linear
    ↓
10 RMSNorm, SwiGLU, Residual
    ↓
11 Causal Attention
    ↓
12 RoPE + Transformer Blocks + logits (B,T,V)
    ↓
13 Cross Entropy + AdamW + LR + Clipping
    ↓
14 Data loader + training + checkpoint + sampling
    ↓
15 Resource Accounting + integration checklist
```

这是一条**从文本到训练和评估**的全路径；只学完 Transformer 结构不足以完成 Assignment 1。


## 2. 模块接口与典型 shape

| 模块 | 输入 | 输出 | 首要 invariant |
|---|---|---|---|
| Pre-tokenization | `str` | `list[str]` | 拼回原 ordinary span |
| BPE Trainer | `PretokenCounts` | `vocab, ordered merges` | 全局 winner，确定性 tie-break |
| Tokenizer | `str` | `list[int]` | `decode(encode(text)) == text` |
| Embedding | `(B,T)` IDs | `(B,T,D)` | 相同 ID 查询同一行 |
| RMSNorm | `(B,T,D)` | `(B,T,D)` | 沿 D 维归一化 |
| Causal MHA | `(B,T,D)` | `(B,T,D)` | 未来 token 不影响前缀 |
| Transformer LM | `(B,T)` IDs | `(B,T,V)` | 每位置 logits |
| CE Loss | logits + targets | scalar | 稳定计算，正确 shift |
| AdamW | params + grads | updated params | 动量与 state 可恢复 |

下面让 Python 计算一个具象配置的 shape 表，避免凭感觉写张量维度。


In [1]:
from __future__ import annotations

import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(336)
torch.set_num_threads(2)


def shape_table(batch, length, d_model, heads, vocab):
    if d_model % heads:
        raise ValueError("d_model must be divisible by heads")
    return {
        "token_ids": (batch, length),
        "hidden": (batch, length, d_model),
        "q_heads": (batch, heads, length, d_model // heads),
        "attention_scores": (batch, heads, length, length),
        "logits": (batch, length, vocab),
        "next_token_labels": (batch, length - 1),
    }


for name, shape in shape_table(2, 8, 32, 4, 128).items():
    print(f"{name:>19}: {shape}")
assert shape_table(2, 8, 32, 4, 128)["q_heads"] == (2, 4, 8, 8)


          token_ids: (2, 8)
             hidden: (2, 8, 32)
            q_heads: (2, 4, 8, 8)
   attention_scores: (2, 4, 8, 8)
             logits: (2, 8, 128)
  next_token_labels: (2, 7)


## 3. Transformer 参数量：从结构推导

考虑一个教学版 decoder-only LM，假设：

- token embedding `(V,D)`；
- `L` 个 block；每个 block 包含无 bias 的 Q/K/V/O 投影；
- SwiGLU hidden size `F`，三个无 bias Linear；
- 每个 block 两个 RMSNorm gain `(D,)`；
- 最后一个 RMSNorm `(D,)` 和**不共享权重**的 LM head `(D,V)`。

那么：

$$
P_{\mathrm{block}}=4D^2+3DF+2D
$$

$$
P_{\mathrm{total}}=VD+L P_{\mathrm{block}}+D+DV.
$$

若使用 weight tying、bias、额外位置 embedding、不同 FFN 比例或特殊结构，公式需要修改。


In [2]:
class CountingBlock(nn.Module):
    def __init__(self, d, f):
        super().__init__()
        self.qkv = nn.Linear(d, 3 * d, bias=False)
        self.o = nn.Linear(d, d, bias=False)
        self.w1 = nn.Linear(d, f, bias=False)
        self.w3 = nn.Linear(d, f, bias=False)
        self.w2 = nn.Linear(f, d, bias=False)
        self.norm1_gain = nn.Parameter(torch.ones(d))
        self.norm2_gain = nn.Parameter(torch.ones(d))


class CountingModel(nn.Module):
    def __init__(self, v, d, f, layers):
        super().__init__()
        self.embedding = nn.Embedding(v, d)
        self.blocks = nn.ModuleList([CountingBlock(d, f) for _ in range(layers)])
        self.final_norm_gain = nn.Parameter(torch.ones(d))
        self.head = nn.Linear(d, v, bias=False)


def parameter_formula(v, d, f, layers):
    return 2 * v * d + layers * (4 * d * d + 3 * d * f + 2 * d) + d


V, D, F_DIM, L = 1000, 64, 176, 4
account_model = CountingModel(V, D, F_DIM, L)
observed = sum(p.numel() for p in account_model.parameters())
predicted = parameter_formula(V, D, F_DIM, L)
assert observed == predicted
print("formula parameters:", predicted)
print("nn.Module parameters:", observed)
print("参数计数对照：通过")


formula parameters: 329280
nn.Module parameters: 329280
参数计数对照：通过


## 4. FLOPs：先理解矩阵乘法成本

一个 `(M,K) @ (K,N)` 的 dense matmul，大致进行 $2MKN$ 次浮点操作（乘加算两次 FLOPs）。

用 `B,T,D,F` 估算每个 block 的**前向**主要成本：

- Q/K/V/O 四次投影：约 $8BTD^2$
- SwiGLU 三次投影：约 $6BTDF$
- QKᵀ 与 Attention×V：约 $4BT^2D$

合起来：

$$
\mathrm{FLOPs}_{\mathrm{forward}}
\approx L\left(8BTD^2+6BTDF+4BT^2D\right)
$$

忽略了 normalization、activation、softmax、head projection、embedding 与 mask 的具体实现差异。由于 causal mask 的稀疏区域并不一定被 kernel 省掉，这个公式是**数量级估算**，不是硬件实际操作数的承诺。


In [3]:
def approximate_forward_flops(batch, seq_len, d_model, d_ff, layers):
    projection = 8 * batch * seq_len * d_model**2
    feedforward = 6 * batch * seq_len * d_model * d_ff
    attention = 4 * batch * seq_len**2 * d_model
    return {
        "projection": layers * projection,
        "feedforward": layers * feedforward,
        "attention": layers * attention,
        "total": layers * (projection + feedforward + attention),
    }


flops = approximate_forward_flops(batch=2, seq_len=128, d_model=D, d_ff=F_DIM, layers=L)
for name, n in flops.items():
    print(f"{name:>12}: {n:,} FLOPs (approx.)")
assert flops["total"] == sum(
    flops[k] for k in ("projection", "feedforward", "attention")
)


  projection: 33,554,432 FLOPs (approx.)
 feedforward: 69,206,016 FLOPs (approx.)
   attention: 33,554,432 FLOPs (approx.)
       total: 136,314,880 FLOPs (approx.)


## 5. 为什么 context length 增大会带来成本？

在固定 `B,D,F,L` 下：

- 参数矩阵乘法成本与 `T` 近似成正比；
- 朴素 attention score 的形状 `(B,H,T,T)`，存储与计算含有 $T^2$ 项。

因此 `T` 增长时，attention 的相对占比往往上升；这也是后面 Assignment 2 学习高效 Attention/Kernels 的动机之一。


In [4]:
for context in (64, 128, 256):
    pieces = approximate_forward_flops(2, context, D, F_DIM, L)
    share = pieces["attention"] / pieces["total"]
    print(
        f"sequence {context:>3}: total={pieces['total']:,}, attention share={share:.1%}"
    )
assert approximate_forward_flops(2, 256, D, F_DIM, L)["attention"] == (
    4 * approximate_forward_flops(2, 128, D, F_DIM, L)["attention"]
)
print("T 翻倍，朴素 Attention 项增长四倍：通过")


sequence  64: total=59,768,832, attention share=14.0%
sequence 128: total=136,314,880, attention share=24.6%
sequence 256: total=339,738,624, attention share=39.5%
T 翻倍，朴素 Attention 项增长四倍：通过


## 6. 训练显存：不仅是模型参数

若**参数与梯度都是 FP32，Adam 一/二阶动量也都是 FP32**，忽略 master weights、碎片等额外开销时：

```text
parameters : 4 bytes / parameter
gradients  : 4 bytes / parameter
Adam moments m,v : 8 bytes / parameter
-------------------------------------
parameter-related total: 16 bytes / parameter
```

这还**不包含**：activations、attention weights、临时缓冲、数据 batch、CUDA context、allocator overhead 等。混合精度和 sharding 会改变账本。

显存估算必须先写出 dtype 与训练策略假设。


In [5]:
num_params = predicted
parameter_bytes = num_params * 4
gradient_bytes = num_params * 4
optimizer_bytes = num_params * 8
rough_total = parameter_bytes + gradient_bytes + optimizer_bytes
assert rough_total == num_params * 16
print("parameters (MiB):", round(parameter_bytes / 2**20, 3))
print("gradients  (MiB):", round(gradient_bytes / 2**20, 3))
print("Adam state (MiB):", round(optimizer_bytes / 2**20, 3))
print("subtotal   (MiB):", round(rough_total / 2**20, 3))
print("注意：不包含 activations / buffers / CUDA runtime")


parameters (MiB): 1.256
gradients  (MiB): 1.256
Adam state (MiB): 2.512
subtotal   (MiB): 5.024
注意：不包含 activations / buffers / CUDA runtime


## 7. 激活显存与训练吞吐量

朴素 attention weights 张量的元素量为 `B*H*T*T`（每层）。若对所有 `L` 层保存 FP32 weights，可据此得到一个粗略的内存下界/成本构成；不同 backward 实现及 fused kernels 的实际保存策略会不同。

训练 throughput 常按 `tokens/sec` 统计，单个 step 的 token 数是 `B*T`，若执行 `S` 步，**未考虑梯度累积时**处理的 token 总数是 `B*T*S`。


In [6]:
B, H, T, L = 2, 4, 128, 4
score_elements = L * B * H * T * T
score_bytes = score_elements * 4
steps = 500
tokens_processed = B * T * steps
print("naive attention score storage (MiB):", round(score_bytes / 2**20, 3))
print("token positions processed:", tokens_processed)
assert tokens_processed == (B * T) * steps


naive attention score storage (MiB): 2.0
token positions processed: 128000


## 8. 从 07 到 14 的跨模块不变量

一个系统可以全部运行而仍然有错误。更可靠的是让每个连接处都存在可单独检验的 invariant：

1. **Tokenizer**：UTF-8 round-trip；special tokens atomic；ordered merges。
2. **File-based statistics**：全文件 Counter 与安全分块聚合一致。
3. **MHA**：输出 shape 正确；修改未来输入不影响过去。
4. **Loss**：与稳定参考实现数值及梯度相同。
5. **Optimizer**：固定步数下更新结果与参考一致。
6. **Checkpoint**：加载后 logits、optimizer states 一致。
7. **Training**：固定可学习合成数据上 loss 确实下降。

它们共同形成了比单个训练 loss 更可靠的测试体系。


In [7]:
# 三个独立的小型核对：byte round-trip、张量形状、CE 数值等价。
sample = "hello 你🌍<|endoftext|>"
roundtrip = sample.encode("utf-8").decode("utf-8")
assert roundtrip == sample

ids = torch.randint(0, 50, (2, 4))
emb = nn.Embedding(50, 12)
hidden = emb(ids)
assert hidden.shape == (2, 4, 12)

logits = torch.randn(2, 4, 50)
labels = torch.randint(0, 50, (2, 4))
ce1 = F.cross_entropy(logits.reshape(-1, 50), labels.reshape(-1))
shift = logits - logits.max(dim=-1, keepdim=True).values
ce2 = (
    torch.logsumexp(shift, dim=-1) - shift.gather(-1, labels.unsqueeze(-1)).squeeze(-1)
).mean()
torch.testing.assert_close(ce1, ce2, rtol=1e-6, atol=1e-6)
print("Encoding + Embedding + Cross Entropy 接口核对：通过")


Encoding + Embedding + Cross Entropy 接口核对：通过


## 9. 从 Notebook 过渡到官方 Assignment 1

完成本章后，正式作业的合理顺序为：

```text
Clone official assignment1-basics
→ read exact handout + tests/adapters.py
→ set up official environment (uv/lockfile)
→ implement your own .py modules
→ connect adapters
→ run focused pytest, then full pytest
→ train/evaluate TinyStories and other required datasets
→ log results and resource accounting
```

Spring 2026 的具体版本、测试接口、实验要求可能更新，所以迁移工程时应以当时仓库中的文档为准，不能把本教材的教学接口当作官方固定签名。
